In [43]:
import torch
from torch import nn, optim # optim：优化器

In [44]:
# 自定义模型
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        # 全连接层
        self.linear = nn.Linear(5, 3) # 输入特征数为5，输出特征数为3
        # 初始化权重
        self.linear.weight.data = torch.tensor(
            [
                [0.1, 0.2, 0.3],
                [0.4, 0.5, 0.6],
                [0.7, 0.8, 0.9],
                [1.0, 1.1, 1.2],
                [1.3, 1.4, 1.5],
            ]
        ).T
        # 初始化偏置
        self.linear.bias.data = torch.tensor([1.0, 2.0, 3.0])

    # 前向传播
    def forward(self, x):
        y = self.linear(x)
        return y

In [45]:
# 1. 定义数据
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
# 目标值，三分类独热编码后的分类标签
target = torch.tensor([0, 0, 1]).float()

In [46]:
# 2. 定义模型
model = MyModel()
print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[0.1000, 0.4000, 0.7000, 1.0000, 1.3000],
        [0.2000, 0.5000, 0.8000, 1.1000, 1.4000],
        [0.3000, 0.6000, 0.9000, 1.2000, 1.5000]], requires_grad=True)
Parameter containing:
tensor([1., 2., 3.], requires_grad=True)


In [47]:
# 3. 训练模型
# 3.1 前向传播， 计算输出值
output = model(x)
print("输出值: ", output)

输出值:  tensor([14.5000, 17.0000, 19.5000], grad_fn=<ViewBackward0>)


In [48]:
# 3.2 计算损失
# loss_fn = nn.MSELoss() # 均方误差损失函数
loss_fn = nn.CrossEntropyLoss() # 交叉熵损失函数
loss = loss_fn(output, target)
print("损失值: ", loss)

损失值:  tensor(0.0851, grad_fn=<DivBackward1>)


In [49]:
# 对于CrossEntropyLoss，计算分类输出概率
proba = torch.softmax(output, dim=-1)
print(proba)

tensor([0.0062, 0.0754, 0.9184], grad_fn=<SoftmaxBackward0>)


In [50]:
# 3.3 反向传播，计算梯度
loss.backward()

In [51]:
print(model.linear.weight.grad)
print(model.linear.bias.grad)

tensor([[ 0.0062,  0.0124,  0.0186,  0.0248,  0.0309],
        [ 0.0754,  0.1508,  0.2262,  0.3016,  0.3769],
        [-0.0816, -0.1632, -0.2447, -0.3263, -0.4079]])
tensor([ 0.0062,  0.0754, -0.0816])


In [57]:
# MSE
# E = 2 / 3 * (output - target).reshape(1, -1) # reshape(1, -1) 将输出值和目标值的差值转换为二维张量
# X = x.reshape(1, -1)
# grad_W = X.T @ E # 梯度计算公式：grad_W = X.T @ E
# print("权重梯度: ", grad_W)

# CrossEntropyLoss
E = (proba - target).reshape(1, -1)
X = x.reshape(1, -1)
grad_W = X.T @ E # 梯度计算公式：grad_W = X.T @ E
print("权重梯度: ", grad_W)

tensor([[ 0.0062,  0.0754, -0.0816],
        [ 0.0124,  0.1508, -0.1632],
        [ 0.0186,  0.2262, -0.2447],
        [ 0.0248,  0.3016, -0.3263],
        [ 0.0309,  0.3769, -0.4079]], grad_fn=<MmBackward0>)


In [53]:
# 3.4 更新参数（迭代1次）
optimizer = optim.SGD(model.parameters(), lr=0.1) # 定义优化器，使用随机梯度下降法
optimizer.step()


In [54]:
print("更新后的权重: ", model.linear.weight)

更新后的权重:  Parameter containing:
tensor([[0.0994, 0.3988, 0.6981, 0.9975, 1.2969],
        [0.1925, 0.4849, 0.7774, 1.0698, 1.3623],
        [0.3082, 0.6163, 0.9245, 1.2326, 1.5408]], requires_grad=True)


In [55]:
# 3.5 梯度清零
optimizer.zero_grad()

In [56]:
print(model.linear.weight.grad)

None
